# PHISIX API (Rust Port) — Endpoint Demo

This notebook demonstrates every endpoint exposed by the **PHISIX API**, a Rust port of the Philippine Stock Exchange Composite Index (PSEi) RESTful API.

The service scrapes live data from `https://frames.pse.com.ph`, serves it as JSON or JAXB-compatible XML, and archives history into a local SQLite database.

## Endpoints covered

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/stocks` | All live stocks (JSON default) |
| `GET` | `/stocks.json` | All live stocks as JSON |
| `GET` | `/stocks.xml` | All live stocks as XML |
| `GET` | `/stocks/{symbol}` | Single live stock (e.g. `/stocks/ali`) |
| `GET` | `/stocks/{symbol}.xml` | Single live stock as XML |
| `GET` | `/stocks/{symbol}.{date}` | Historical lookup (DB, then proxy fallback) |
| `GET` / `POST` | `/stocks/archive` | Archive current feed to SQLite |

## Prerequisites

Start the server first (default port `8080`):

```bash
cargo run
# or containerized:
./build.sh && ./run.sh
```

This notebook only needs the `requests` library:

```bash
pip install requests
```

## Setup

Configure the base URL and a couple of small helpers to pretty-print responses. Change `BASE_URL` if you run the server on a different host or port.

In [ ]:
import json
import xml.dom.minidom

import requests

BASE_URL = "http://localhost:8080"


def show(resp):
    """Print status, content-type, and a pretty-printed body (JSON or XML)."""
    ctype = resp.headers.get("content-type", "")
    print(f"{resp.request.method} {resp.url}")
    print(f"-> {resp.status_code} {resp.reason}  |  content-type: {ctype}\n")

    if "json" in ctype:
        print(json.dumps(resp.json(), indent=2))
    elif "xml" in ctype:
        print(xml.dom.minidom.parseString(resp.text).toprettyxml(indent="  ").strip())
    else:
        print(resp.text)


# Quick reachability check
try:
    ping = requests.get(f"{BASE_URL}/stocks", timeout=15)
    print(f"Server reachable: {ping.status_code} {ping.reason}")
except requests.exceptions.RequestException as exc:
    print(f"Could not reach {BASE_URL} — is the server running? ({exc})")

## 1. Get all live stocks (JSON)

`GET /stocks` returns the full PSEi feed as JSON. The server caches the scraped feed for **60 seconds**, so repeated calls within that window are served from memory.

The response shape is:

```jsonc
{
  "as_of": "2026-07-03T15:20:00+08:00",   // ISO 8601, Asia/Manila (GMT+8)
  "stocks": [
    {
      "name": "Ayala Land, Inc.",
      "symbol": "ALI",
      "price": { "currency": "PHP", "amount": 30.5 },
      "percent_change": 0.66,
      "volume": 12345678
    }
  ]
}
```

In [ ]:
resp = requests.get(f"{BASE_URL}/stocks", timeout=30)

data = resp.json()
print(f"as_of: {data['as_of']}")
print(f"total stocks: {len(data['stocks'])}\n")

# Show the first 5 entries
print(json.dumps(data["stocks"][:5], indent=2))

### `/stocks.json` — explicit JSON suffix

Identical payload to `/stocks`; the `.json` suffix is supported for compatibility with the original API.

In [ ]:
resp = requests.get(f"{BASE_URL}/stocks.json", timeout=30)
print(f"{resp.status_code} {resp.reason}  |  content-type: {resp.headers.get('content-type')}")
print(f"total stocks: {len(resp.json()['stocks'])}")

## 2. Get all live stocks (XML)

`GET /stocks.xml` returns the same data serialized as JAXB-compatible XML using the `stocks:` namespace, matching the original Java implementation byte-for-byte.

In [ ]:
resp = requests.get(f"{BASE_URL}/stocks.xml", timeout=30)
print(f"{resp.status_code} {resp.reason}  |  content-type: {resp.headers.get('content-type')}\n")

# Print just the header + first stock block to keep output short
pretty = xml.dom.minidom.parseString(resp.text).toprettyxml(indent="  ")
print("\n".join(pretty.splitlines()[:14]))
print("  ...")

## 3. Single stock lookup

`GET /stocks/{symbol}` returns just one company from the live feed. The symbol is case-insensitive (`ali`, `ALI`, and `Ali` all work).

A missing symbol returns **404 Not Found**.

In [ ]:
SYMBOL = "ali"  # Ayala Land, Inc.

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}", timeout=30)
show(resp)

### Same stock as XML

Append `.xml` to any single-stock path to get the XML representation (`.json` is also accepted).

In [ ]:
resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.xml", timeout=30)
show(resp)

### Unknown symbol → 404

In [ ]:
resp = requests.get(f"{BASE_URL}/stocks/NOSUCHSYMBOL", timeout=30)
print(f"{resp.status_code} {resp.reason}")
assert resp.status_code == 404

## 4. Historical stock lookup

`GET /stocks/{symbol}.{date}` looks up a stock on a specific trading date (`YYYY-MM-DD`).

The resolution order is:
1. **Local SQLite** — return the archived record if present.
2. **Proxy fallback** — otherwise fetch from `https://phisix-api2.appspot.com/stocks/{SYMBOL}.{date}.json`, return it, and **cache it locally** for next time.
3. If neither has data, return **404 Not Found**.

Both `.json` and `.xml` suffixes are supported on historical paths too.

In [ ]:
DATE = "2023-09-03"  # YYYY-MM-DD

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{DATE}", timeout=30)
show(resp)

In [ ]:
# Historical lookup as XML
resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{DATE}.xml", timeout=30)
show(resp)

## 5. Archive current feed

`GET` or `POST /stocks/archive` fetches the current live feed and saves/updates it in the local SQLite database. This is typically wired to a cron job so that daily snapshots accumulate for later historical queries.

Returns `200 OK` with the plain-text body `Archived successfully`.

In [ ]:
# GET form
resp = requests.get(f"{BASE_URL}/stocks/archive", timeout=30)
print(f"GET  -> {resp.status_code} {resp.reason}: {resp.text!r}")

# POST form (equivalent)
resp = requests.post(f"{BASE_URL}/stocks/archive", timeout=30)
print(f"POST -> {resp.status_code} {resp.reason}: {resp.text!r}")

### Confirm the archive worked

After archiving today's feed, a historical lookup for **today's date** should now resolve from the local database instead of the proxy.

In [ ]:
import datetime
import zoneinfo

# The API uses Asia/Manila (GMT+8) for trading dates
today_manila = datetime.datetime.now(zoneinfo.ZoneInfo("Asia/Manila")).strftime("%Y-%m-%d")
print(f"Today (Asia/Manila): {today_manila}\n")

resp = requests.get(f"{BASE_URL}/stocks/{SYMBOL}.{today_manila}", timeout=30)
show(resp)

## 6. Bonus — load the feed into a DataFrame

The JSON structure maps cleanly onto a tabular view for quick analysis. Requires `pandas` (`pip install pandas`).

In [ ]:
import pandas as pd

data = requests.get(f"{BASE_URL}/stocks", timeout=30).json()

df = pd.json_normalize(data["stocks"])
df = df.rename(columns={"price.currency": "currency", "price.amount": "amount"})

print(f"Feed as_of: {data['as_of']}  |  {len(df)} stocks\n")

# Top 10 gainers by percent change
df.sort_values("percent_change", ascending=False).head(10)

---

That covers every endpoint the PHISIX API exposes. See the accompanying **Bruno** collection under `bruno/` for the same requests in an API-client format.